# Ensemble atmosphere-land cross-correlation

Reads paired EAM initial files and ELM restart files, aggregates land columns to grid cells, collocates the land and atmosphere grids, computes ensemble correlations, caches each request, and plots the selected diagnostics.


In [ ]:
from pathlib import Path
INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_out")

case_name = "DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy"
time_tag = "2011-12-18-00000"
atmosphere_requests = ({"name": "T", "level": 79, "dim": "lev"},)
land_requests = ({"name": "T_SOISNO", "level": 6, "dim": "levtot"},)
mode = "collocated"  # "collocated" or "landpoint_all_atm"
land_point_index = 0
ensemble_ids = None
active_land_columns_only = True
plot_products = ("map", "histogram", "latitude_mean")
force_compute = False
show = True
verbose = True


In [ ]:
import sys
repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))

from configs.output_paths import workflow_output_dirs
from util.lnd_atm_cross_correlation import (
    VariableSpec,
    AtmosLandCrossCorrelationAnalyzer,
    CrossCorrelationPlotter,
    correlation_cache_path,
    cache_cross_correlation,
)


## Notes: available variables and diagnostics

Atmosphere variables in the EAM initial files are `PS` (2-D) and `Q`, `T`, `U`, and `V` on the `lev` dimension. Land restart variables are `DZSNO`, `ZISNO`, and `ZSNO` on `levsno`, plus `H2OSOI_ICE`, `H2OSOI_LIQ`, and `T_SOISNO` on `levtot`. Use `level=None` and `dim=None` for a 2-D variable.

The cached metrics are ensemble `correlation`, land-atmosphere `distance_km`, mean land and atmosphere values, coordinates, and collocation indices. Plot products are `map`, `histogram`, `latitude_mean`, `distance_scatter`, `distance_mean`, `distance_percentiles`, and `distance_exceedance`.

`collocated` correlates every land grid cell with its nearest atmosphere column. `landpoint_all_atm` correlates one selected land grid cell against all atmosphere columns. Existing NetCDF diagnostics are reused unless `force_compute = True`.


In [ ]:
from pathlib import Path

data_dir, figure_dir = workflow_output_dirs(
    "analysis_lac", "cross_correlation", output_root=DIAGNOSTIC_OUTPUT_ROOT
)
case_dir = INPUT_DATA_ROOT / case_name
requests = [
    (
        VariableSpec(item_atm["name"], item_atm["level"], item_atm["dim"]),
        VariableSpec(item_lnd["name"], item_lnd["level"], item_lnd["dim"]),
    )
    for item_atm in atmosphere_requests
    for item_lnd in land_requests
]
targets = [
    correlation_cache_path(
        data_dir, time_tag, atm.name, atm.level_index,
        land.name, land.level_index, mode,
    )
    for atm, land in requests
]
missing = [path for path in targets if force_compute or not path.is_file()]
analyzer = None
if missing:
    analyzer = AtmosLandCrossCorrelationAnalyzer(
        base_dir=str(case_dir), time_tag=time_tag, ensemble_ids=ensemble_ids
    )
    for atm, land in requests:
        analyzer.filter_valid_ensembles(
            atm_var=atm, land_var=land,
            active_only=active_land_columns_only, verbose=verbose,
        )
    if not analyzer.eam_files:
        raise RuntimeError("No ensemble pairs remain after variable validation")
    analyzer.build_land_grid(requests[0][1], active_only=active_land_columns_only)
    analyzer.build_land_to_atm_mapping()

for (atm, land), target in zip(requests, targets):
    existed = target.is_file()
    ds = cache_cross_correlation(
        target,
        analyzer=analyzer,
        atm_var=atm,
        land_var=land,
        mode=mode,
        land_point_index=land_point_index,
        active_only=active_land_columns_only,
        force_compute=force_compute,
    )
    print("[REUSE]" if existed and not force_compute else "[WRITE]", target)
    ds.close()

    plotter = CrossCorrelationPlotter(target)
    tag = plotter.get_tag(time_tag=time_tag)
    products = {
        "map": lambda: plotter.plot_global_scatter(
            outfile=figure_dir / f"correlation_map_{mode}_{tag}.png", show=show
        ),
        "histogram": lambda: plotter.plot_histogram(
            outfile=figure_dir / f"correlation_histogram_{mode}_{tag}.png", show=show
        ),
        "latitude_mean": lambda: plotter.plot_latitudinal_mean(
            outfile=figure_dir / f"correlation_latitude_mean_{mode}_{tag}.png", show=show
        ),
        "distance_scatter": lambda: plotter.plot_distance_relationship(
            outfile=figure_dir / f"correlation_distance_scatter_{mode}_{tag}.png", show=show
        ),
        "distance_mean": lambda: plotter.plot_binned_distance_mean(
            bins=25, use_abs=True,
            outfile=figure_dir / f"correlation_distance_mean_{mode}_{tag}.png", show=show
        ),
        "distance_percentiles": lambda: plotter.plot_binned_distance_percentiles(
            bins=25, use_abs=True,
            outfile=figure_dir / f"correlation_distance_percentiles_{mode}_{tag}.png", show=show
        ),
        "distance_exceedance": lambda: plotter.plot_distance_exceedance(
            bins=25, threshold=0.2,
            outfile=figure_dir / f"correlation_distance_exceedance_{mode}_{tag}.png", show=show
        ),
    }
    unknown = set(plot_products) - set(products)
    if unknown:
        raise ValueError(f"Unknown plot products: {sorted(unknown)}")
    for product in plot_products:
        products[product]()
    plotter.close()
